# Part 8 · Notebook 08 — Robustness tests and the validation gate

**Sessions:** S15 (Robustness tests) · S16 (The validation gate & M4 release) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Test whether the result survives without its best days.
2. Perturb every parameter and check the Sharpe holds.
3. Combine everything into a pass/fail validation gate, and see a decent strategy fail it.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. The candidate

TSMOM on the known-regime market, lookback 160 and vol window 60 (the plateau from notebook 05). `run(params, delay, cost_mult)` returns its daily P&L, optionally with the signal delayed and the costs multiplied: every robustness test is a variation of it.

In [ ]:
bars = p.regime_market()
o, c = bars.open.to_numpy(), bars.close.to_numpy()

def run(params, delay=0, cost_mult=1.0):
    sig = p.tsmom_signal(c, params["lookback"], params["vol_n"])
    if delay:
        sig = np.r_[np.zeros(delay), sig[:-delay]]
    return p.first_look_pnl(sig, o, cost_bps=2.0 * cost_mult)

params = {"lookback": 160, "vol_n": 60}
base = run(params)
print(f"base Sharpe {p.sharpe(base):.2f}")

## 2. Without the best days

If a handful of days make the whole result, it is a lottery ticket. Remove the `k` largest daily P&Ls and recompute the Sharpe (`np.argsort` gives the order; `np.delete` removes by index).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def sharpe_without_best(pnl, k=5):
    pnl = np.asarray(pnl, dtype=float)
    rest = ...                                    # ✍️ pnl with its k largest values removed
    return p.sharpe(rest)

ks = [1, 5, 20, 50]
mine = [p.attempt(sharpe_without_best, base, k) for k in ks]
mine = p.check("sharpe_without_best", mine, [p.sharpe(np.delete(base, np.argsort(base)[-k:])) for k in ks])
pd.Series(mine, index=[f"without best {k}" for k in ks]).round(2)

## 3. Parameter perturbation

For every **integer** parameter, rerun with it scaled by `1 − perturb` and `1 + perturb` (rounded to an int), keeping the others. Return `(test name, Sharpe)` pairs named like `'lookback -20%'`, `'lookback +20%'`, in the dict's order, minus before plus.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def perturbation_tests(run, params, perturb=0.2):
    out = []
    for k, v in params.items():
        for sgn, lab in ((-1, "-"), (1, "+")):
            p2 = ...                              # ✍️ params with k replaced by int(round(v × (1 ± perturb)))
            out.append((f"{k} {lab}{int(perturb * 100)}%", p.sharpe(run(p2))))
    return out

mine = p.attempt(perturbation_tests, run, params)
card = p.robustness_scorecard(run, params)
mine = p.check("perturbation_tests", mine, [(t, v) for t, v, _ in card.itertuples(index=False)][:4])
card.round(2)

The full scorecard adds a one-bar delay, doubled costs, and each half of the history on its own. TSMOM passes all nine.

## 4. The validation gate

A strategy is promoted to paper trading only if it passes **every** check (`p.GATE` holds the thresholds):

| Check | Pass if |
|---|---|
| out-of-sample Sharpe (walk-forward) | >= 0.5 |
| Deflated Sharpe of the OOS P&L, given all trials | >= 0.95 |
| PBO of the whole search | <= 0.2 |
| share of robustness tests passed | >= 0.8 |
| OOS max drawdown | >= −25% |

Build the evidence: the 48-trial grid from notebook 05 (for the trial Sharpes and PBO), a walk-forward OOS series, and the scorecard.

In [ ]:
grid = [dict(lookback=lb, vol_n=vn) for lb in (20, 40, 60, 90, 120, 160, 200, 250) for vn in (10, 20, 40, 60, 90, 120)]
perf = np.column_stack([run(g) for g in grid])
trial_srs = perf.mean(0) / perf.std(0, ddof=1)
pbo, _ = p.pbo_cscv(perf, S=10)
wf = p.walk_forward_optimize(lambda lookback, vol_n: run(dict(lookback=lookback, vol_n=vol_n)),
                             {"lookback": [40, 90, 160, 250], "vol_n": [20, 60]}, len(c), 750, 250)
print(f"{len(grid)} trials, PBO {pbo:.2f}, walk-forward OOS Sharpe {p.sharpe(wf['oos']):.2f}")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

Write the gate: compute each check's value and whether it passes, and `passed = all of them`. Return `{"checks": {name: (value, passed)}, "passed": bool}` with the names `oos_sharpe`, `dsr`, `pbo`, `robustness`, `max_dd`.

In [ ]:
def validation_gate(oos_pnl, trial_srs, pbo, scorecard, c=p.GATE):
    r = np.asarray(oos_pnl, dtype=float)
    oos_sr = p.sharpe(r)
    dsr, _ = p.deflated_sharpe(r, trial_srs)
    rob = float(scorecard["passed"].mean())
    mdd = p.max_drawdown(r)[0]
    checks = {"oos_sharpe": (oos_sr, oos_sr >= c["min_oos_sharpe"]),
              "dsr": ...,                         # ✍️
              "pbo": ...,                         # ✍️
              "robustness": (rob, rob >= c["min_robustness"]),
              "max_dd": (mdd, mdd >= c["max_drawdown"])}
    return {"checks": checks, "passed": all(ok for _, ok in checks.values())}

mine = p.attempt(validation_gate, wf["oos"], trial_srs, pbo, card)
mine = p.check("validation_gate", mine, p.validation_gate(wf["oos"], trial_srs, pbo, card))
display(pd.DataFrame(mine["checks"], index=["value", "passed"]).T)
print("VERDICT:", "PASS" if mine["passed"] else "FAIL")

A strategy with a real mechanism (it earns in trends by construction), a 0.9 out-of-sample Sharpe and a perfect robustness scorecard still **fails**: six years of out-of-sample data after 48 trials isn't enough evidence (DSR just under 0.95), and the search ranking is not stable enough (PBO above 0.2). The right response is more evidence (more markets, more history, paper trading), not loosening the gate until it passes.

## Wrap-up

* Robustness: without the best days, parameters ±20%, a bar of delay, doubled costs, each half alone.
* The gate is a pre-registered, all-must-pass rule; write it down before you look at the results.
* Graded versions: `labs/part08/week28_overfitting` (scorecard, gate, dossier) and Clinic W4 (validating three strategies).